# Karaokê Zapia — processamento no Kaggle GPU

O notebook aceita um link compartilhado do MukaPlay (vídeo MP4) ou um arquivo de áudio/vídeo que você tenha autorização para processar. Ele extrai o áudio do vídeo, separa a voz com Demucs, sincroniza a letra com Whisper e gera instrumental, SRT e MP4.

**Respeite os direitos autorais e as regras da plataforma de origem.** Só use conteúdo criado por você ou que tenha autorização para baixar e processar. Marque essa confirmação na célula de configuração. O notebook acessa o link do MukaPlay para baixar o MP4; não extrai áudio diretamente do player do YouTube.

O Kaggle oferece GPU em sessões de notebook; a cota é semanal e pode variar. Ative **GPU** e **Internet** em Settings. Para uma faixa, o notebook pode baixar o vídeo do MukaPlay diretamente, sem precisar enviar o MP4 como arquivo separado.

Documentação do Kaggle: [uso eficiente de GPU](https://www.kaggle.com/docs/efficient-gpu-usage).

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install demucs faster-whisper requests
import torch
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## Worker Demucs + Whisper + FFmpeg

A célula grava o worker do projeto no ambiente do notebook.

In [ ]:
%%writefile /kaggle/working/gpu_worker.py
#!/usr/bin/env python3
"""Extract authorized audio/video input, separate vocals, transcribe, and render a karaoke MP4.

Video files may be supplied from an authorized source such as a MukaPlay share link
resolved by the caller. The worker itself does not scrape YouTube.
"""
from __future__ import annotations

import argparse
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path


def run(cmd: list[str]) -> None:
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode:
        detail = (result.stderr or result.stdout or "").strip()[-2500:]
        raise RuntimeError(f"Command failed: {' '.join(cmd[:3])}\n{detail}")


def stamp(seconds: float) -> str:
    ms = max(0, int(round(seconds * 1000)))
    hours, ms = divmod(ms, 3_600_000)
    minutes, ms = divmod(ms, 60_000)
    secs, ms = divmod(ms, 1000)
    return f"{hours:02}:{minutes:02}:{secs:02},{ms:03}"


def safe_srt_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().replace("--> ", "→ ")


def make_captions(segments) -> list[tuple[float, float, str]]:
    """Group Whisper's word timestamps into readable, short karaoke captions."""
    words = []
    fallback = []
    for segment in segments:
        fallback.append((float(segment.start), float(segment.end), safe_srt_text(segment.text)))
        for word in (getattr(segment, "words", None) or []):
            text = safe_srt_text(word.word)
            if text:
                words.append((float(word.start), float(word.end), text))
    if not words:
        return [x for x in fallback if x[2]]

    lines = []
    group = []
    start = words[0][0]
    for word in words:
        if group and (len(group) >= 7 or word[0] - start >= 4.0):
            lines.append((start, group[-1][1], "".join(x[2] for x in group).strip()))
            group = []
            start = word[0]
        if not group:
            start = word[0]
        group.append(word)
        if re.search(r"[.!?;:]$", word[2]) and len(group) >= 3:
            lines.append((start, word[1], "".join(x[2] for x in group).strip()))
            group = []
    if group:
        lines.append((start, group[-1][1], "".join(x[2] for x in group).strip()))
    return [(a, max(a + 0.25, b), text) for a, b, text in lines if text]


def transpose_audio(source: Path, destination: Path, semitones: float) -> None:
    """Shift pitch while approximately preserving duration (FFmpeg atempo/asetrate)."""
    if abs(semitones) < 1e-6:
        shutil.copy2(source, destination)
        return
    try:
        sample_rate = int(subprocess.check_output([
            "ffprobe", "-v", "error", "-select_streams", "a:0",
            "-show_entries", "stream=sample_rate", "-of", "default=noprint_wrappers=1:nokey=1", str(source),
        ], text=True).strip())
    except Exception as exc:
        raise RuntimeError("Não consegui identificar a taxa de amostragem para ajustar o tom.") from exc
    ratio = 2 ** (semitones / 12)
    atempo = 1 / ratio
    audio_filter = f"asetrate={sample_rate * ratio:.5f},aresample={sample_rate},atempo={atempo:.8f}"
    print(f"Ajustando tom: {semitones:+g} semitons (mantendo a duração aproximada)…", flush=True)
    run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(source), "-af", audio_filter, "-c:a", "pcm_s16le", str(destination)])


def main() -> int:
    parser = argparse.ArgumentParser(description="Gera instrumental, legenda e MP4 de karaokê a partir de mídia autorizada.")
    parser.add_argument("audio", type=Path, help="Arquivo de áudio ou vídeo que você tem autorização para processar")
    parser.add_argument("--output-dir", type=Path, default=Path("karaoke-output"))
    parser.add_argument("--demucs-model", default="htdemucs")
    parser.add_argument("--whisper-model", default="small", help="tiny, base, small, medium ou large-v3")
    parser.add_argument("--language", default=None, help="Idioma opcional, por exemplo pt")
    parser.add_argument("--semitones", type=int, default=0, help="Transposição de -6 (mais grave) a +6 (mais agudo)")
    args = parser.parse_args()
    if not -6 <= args.semitones <= 6:
        parser.error("A transposição deve ficar entre -6 e +6 semitons.")

    source = args.audio.expanduser().resolve()
    if not source.is_file():
        parser.error(f"Arquivo não encontrado: {source}")
    audio_extensions = {".mp3", ".wav", ".flac", ".m4a", ".ogg", ".opus"}
    video_extensions = {".mp4", ".m4v", ".mov", ".mkv", ".webm", ".avi"}
    if source.suffix.lower() not in audio_extensions | video_extensions:
        parser.error("Formato não aceito. Use áudio comum ou vídeo MP4/MOV/MKV/WEBM/AVI.")
    if shutil.which("ffmpeg") is None:
        parser.error("FFmpeg não encontrado. Instale FFmpeg antes de rodar.")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    stem = re.sub(r"[^\w.-]+", "_", source.stem, flags=re.UNICODE).strip("._") or "karaoke"

    try:
        import torch
        from faster_whisper import WhisperModel
    except ImportError as exc:
        parser.error(f"Dependência ausente: {exc}. Instale requirements-separation.txt.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    print(f"GPU: {device} | Whisper: {args.whisper_model} ({compute_type})", flush=True)

    with tempfile.TemporaryDirectory(prefix="karaoke_worker_") as temp_name:
        temp = Path(temp_name)
        processing_source = source
        if source.suffix.lower() in video_extensions:
            processing_source = temp / "source_audio.wav"
            print("Extraindo a faixa de áudio do vídeo autorizado com FFmpeg…", flush=True)
            run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(source), "-map", "0:a:0", "-vn", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(processing_source)])
        demucs_out = temp / "separated"
        print("Separando voz e instrumental com Demucs…", flush=True)
        run([sys.executable, "-m", "demucs", "--two-stems=vocals", "-n", args.demucs_model, "-o", str(demucs_out), str(processing_source)])
        instrumental_src = next(demucs_out.rglob("no_vocals.wav"), None)
        if instrumental_src is None:
            raise RuntimeError("Demucs terminou, mas a faixa instrumental não foi encontrada.")
        shifted_src = temp / "pitch_adjusted.wav"
        transpose_audio(instrumental_src, shifted_src, args.semitones)
        instrumental_src = shifted_src

        print("Reconhecendo a letra e os tempos com Whisper…", flush=True)
        model = WhisperModel(args.whisper_model, device=device, compute_type=compute_type)
        segments_iter, _info = model.transcribe(str(processing_source), language=args.language, word_timestamps=True, vad_filter=True)
        segments = list(segments_iter)
        captions = make_captions(segments)

        srt_path = args.output_dir / f"{stem}.srt"
        with srt_path.open("w", encoding="utf-8") as f:
            for i, (start, end, text) in enumerate(captions, 1):
                f.write(f"{i}\n{stamp(start)} --> {stamp(end)}\n{text}\n\n")

        tone_label = "tom_0" if args.semitones == 0 else f"tom_{'mais' if args.semitones > 0 else 'menos'}_{abs(args.semitones)}"
        instrumental = args.output_dir / f"{stem}_{tone_label}_instrumental.wav"
        shutil.copy2(instrumental_src, instrumental)
        video = args.output_dir / f"{stem}_{tone_label}_karaoke.mp4"
        duration_cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(instrumental)]
        try:
            duration = float(subprocess.check_output(duration_cmd, text=True).strip())
        except Exception as exc:
            raise RuntimeError("Não consegui medir a duração do instrumental com ffprobe.") from exc
        subtitle_path = str(srt_path.resolve()).replace("\\", "\\\\").replace(":", "\\:").replace("'", "\\'")
        vf = f"subtitles='{subtitle_path}':force_style='FontName=DejaVu Sans,FontSize=32,PrimaryColour=&H00FFFFFF,OutlineColour=&H00101010,BorderStyle=1,Outline=3,Shadow=0,Alignment=2,MarginV=72'"
        print("Renderizando o vídeo MP4…", flush=True)
        run([
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-f", "lavfi", "-i", f"color=c=0x17131f:s=1280x720:r=30:d={duration:.3f}",
            "-i", str(instrumental), "-vf", vf, "-map", "0:v:0", "-map", "1:a:0",
            "-c:v", "libx264", "-preset", "veryfast", "-pix_fmt", "yuv420p",
            "-c:a", "aac", "-b:a", "192k", "-shortest", "-movflags", "+faststart", str(video),
        ])

    print("Pronto:")
    print(f"  MP4: {video.resolve()}")
    print(f"  Instrumental: {instrumental.resolve()}")
    print(f"  Legenda editável: {srt_path.resolve()}")
    print("Revise a letra e os tempos: reconhecimento de canto pode errar palavras e sincronização.")
    return 0


if __name__ == "__main__":
    try:
        raise SystemExit(main())
    except RuntimeError as exc:
        print(f"ERRO: {exc}", file=sys.stderr)
        raise SystemExit(1)


In [ ]:
%%writefile /kaggle/working/muka_resolver.py
"""Resolve and download an authorized MukaPlay share video."""
from __future__ import annotations

from html import unescape
from html.parser import HTMLParser
from pathlib import Path
from urllib.parse import parse_qs, urlparse
import re
import requests

PAGE_HOSTS = {"mukaplay.com", "www.mukaplay.com"}
DOWNLOAD_HOST = "api.muka-box.com"
MAX_PAGE_BYTES = 2 * 1024 * 1024
DEFAULT_MAX_VIDEO_BYTES = 500 * 1024 * 1024


class MukaLinkError(ValueError):
    pass


class _ShareParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.items = []
        self._item = None
        self._in_left = False

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "div" and "list-item" in (attrs.get("class") or "").split():
            self._item = {"label": "", "href": ""}
        elif self._item is not None and tag == "div" and "left" in (attrs.get("class") or "").split():
            self._in_left = True
        elif self._item is not None and tag == "a":
            self._item["href"] = attrs.get("href", "")

    def handle_data(self, data):
        if self._item is not None and self._in_left:
            self._item["label"] += data

    def handle_endtag(self, tag):
        if tag == "div" and self._in_left:
            self._in_left = False
        elif tag == "div" and self._item is not None and self._item["href"]:
            self.items.append(self._item)
            self._item = None


def validate_share_url(url: str) -> str:
    raw = url.strip()
    if raw.lower().startswith("http://"):
        raw = "https://" + raw[7:]
    elif not raw.lower().startswith("https://"):
        raw = "https://" + raw
    parsed = urlparse(raw)
    host = (parsed.hostname or "").lower()
    if parsed.scheme != "https" or host not in PAGE_HOSTS:
        raise MukaLinkError("Cole um link de compartilhamento do MukaPlay.")
    try:
        port = parsed.port
    except ValueError as exc:
        raise MukaLinkError("O link do MukaPlay tem um formato inválido.") from exc
    if parsed.username or parsed.password or port not in (None, 443):
        raise MukaLinkError("O link do MukaPlay tem um formato inválido.")
    path = parsed.path.rstrip("/")
    if not re.fullmatch(r"/dynps/share/[A-Za-z0-9_%=-]+", path):
        raise MukaLinkError("Esse não parece ser um link de compartilhamento do MukaPlay.")
    return parsed._replace(path=path).geturl()


def resolve_download_url(url: str, session=None) -> str:
    """Read only the share page and return its visible MP4 download URL."""
    url = validate_share_url(url)
    session = session or requests.Session()
    response = session.get(url, timeout=(15, 30), headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    if len(response.content) > MAX_PAGE_BYTES:
        raise MukaLinkError("A página de compartilhamento do MukaPlay é grande demais.")
    final = urlparse(response.url)
    if (final.hostname or "").lower() not in PAGE_HOSTS:
        raise MukaLinkError("O link do MukaPlay redirecionou para um domínio inesperado.")

    parser = _ShareParser()
    parser.feed(response.text)
    candidates = []
    for item in parser.items:
        link = unescape(item["href"]).strip()
        parsed = urlparse(link)
        if parsed.scheme != "https" or (parsed.hostname or "").lower() != DOWNLOAD_HOST:
            continue
        if parsed.path != "/api/dev/redirect/redirect":
            continue
        q = parse_qs(parsed.query)
        if q.get("pkg_name") != ["com.seep.muka"] or q.get("pub") != ["video_share"] or not q.get("ext"):
            continue
        candidates.append((item["label"].strip(), link))

    if not candidates:
        raise MukaLinkError("Não encontrei um link de vídeo disponível nessa página do MukaPlay.")
    # MukaPlay's share page may show multiple resolution labels but currently
    # uses the same redirect URL for them; follow the first available MP4 link.
    return candidates[0][1]


def download_muka_video(url: str, destination: Path, max_bytes: int = DEFAULT_MAX_VIDEO_BYTES) -> Path:
    """Download the public MP4 link offered by MukaPlay, with a size cap."""
    session = requests.Session()
    direct_url = resolve_download_url(url, session=session)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    total = 0
    try:
        with session.get(direct_url, stream=True, allow_redirects=True, timeout=(20, 90), headers={
            "User-Agent": "Mozilla/5.0",
            "Referer": "https://www.mukaplay.com/",
        }) as response:
            response.raise_for_status()
            final = urlparse(response.url)
            if final.scheme != "https":
                raise MukaLinkError("O download do MukaPlay terminou em um endereço não seguro.")
            ctype = (response.headers.get("content-type") or "").lower()
            disposition = (response.headers.get("content-disposition") or "").lower()
            if "text/html" in ctype or "application/json" in ctype:
                raise MukaLinkError("O MukaPlay não entregou um arquivo de vídeo. Confira se o link ainda está válido.")
            if "video/" not in ctype and ".mp4" not in disposition and "octet-stream" not in ctype:
                raise MukaLinkError("A resposta do MukaPlay não parece ser um arquivo MP4.")
            declared = response.headers.get("content-length")
            if declared and int(declared) > max_bytes:
                raise MukaLinkError("O vídeo excede o limite de tamanho permitido.")
            with destination.open("wb") as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    total += len(chunk)
                    if total > max_bytes:
                        raise MukaLinkError("O vídeo excede o limite de tamanho permitido.")
                    f.write(chunk)
    except Exception:
        destination.unlink(missing_ok=True)
        raise
    if total == 0:
        destination.unlink(missing_ok=True)
        raise MukaLinkError("O arquivo do MukaPlay veio vazio.")
    return destination


## Link MukaPlay ou arquivo autorizado

Cole o link de compartilhamento MukaPlay na variável `MUKA_URL`; o notebook baixa o MP4 pelo link de download exibido na página e extrai o áudio. Como alternativa, indique um arquivo de áudio ou vídeo autorizado em `AUDIO_PATH`. Defina o tom em `SEMITONES` (-6 a +6) e marque `AUTHORIZED_TO_PROCESS = True` somente se tiver permissão para salvar e processar o conteúdo.

In [ ]:
from pathlib import Path
import json, shutil, sys, subprocess
from muka_resolver import download_muka_video

# Cole o link de compartilhamento do MukaPlay aqui, ou baixe as configurações no site e adicione o JSON como Kaggle Input.
MUKA_URL = ""
AUDIO_PATH = None  # opcional: caminho de arquivo autorizado em /kaggle/input
SEMITONES = 0  # -6 a +6; negativo = mais grave, positivo = mais agudo
AUTHORIZED_TO_PROCESS = False  # mude para True somente se tiver direito/permissão para salvar e processar o conteúdo
settings_files = sorted(Path("/kaggle/input").rglob("karaoke_settings.json"))
if settings_files:
    settings = json.loads(settings_files[0].read_text(encoding="utf-8"))
    MUKA_URL = settings.get("muka_url", MUKA_URL)
    AUDIO_PATH = settings.get("audio_path", AUDIO_PATH)
    SEMITONES = int(settings.get("semitones", SEMITONES))
    AUTHORIZED_TO_PROCESS = bool(settings.get("authorized", AUTHORIZED_TO_PROCESS))
    print("Configuração importada:", settings_files[0])
if not -6 <= SEMITONES <= 6:
    raise ValueError("SEMITONES precisa ficar entre -6 e +6.")
if not AUTHORIZED_TO_PROCESS:
    raise PermissionError("Confirme que você tem direito/permissão para baixar e processar este conteúdo antes de continuar.")

media_exts = {".mp3", ".wav", ".flac", ".m4a", ".ogg", ".opus", ".mp4", ".m4v", ".mov", ".mkv", ".webm", ".avi"}
if MUKA_URL.strip():
    source = download_muka_video(MUKA_URL.strip(), Path("/kaggle/working/muka_source.mp4"))
    source_label = MUKA_URL.strip()
else:
    available = sorted(p for p in Path("/kaggle/input").rglob("*") if p.is_file() and p.suffix.lower() in media_exts)
    if not available:
        raise FileNotFoundError("Cole um link MukaPlay em MUKA_URL ou adicione um arquivo autorizado em Add Input.")
    print("Arquivos de mídia disponíveis:")
    for p in available: print(p)
    if AUDIO_PATH is None: AUDIO_PATH = str(available[0])
    source = Path(AUDIO_PATH).expanduser().resolve()
    if not source.is_file(): raise FileNotFoundError(f"Arquivo não encontrado: {source}")
    source_label = str(source)
print("Fonte selecionada:", source_label)
print("Arquivo local:", source)
print("Tom escolhido:", f"{SEMITONES:+d} semitons")

O worker usa Whisper `tiny` para reduzir o tempo e ajusta o tom selecionado de -6 (mais grave) a +6 (mais agudo), mantendo a duração aproximadamente. Você pode escolher o tom no site, baixar `karaoke_settings.json` e adicioná-lo ao mesmo dataset privado do áudio; ou mudar `SEMITONES` na célula anterior. Revise o SRT: voz cantada pode ser transcrita com erros. Depois, baixe `karaoke_kaggle_output.zip` na área de arquivos de saída do notebook.

In [ ]:
output_dir = Path("/kaggle/working/karaoke-output")
output_dir.mkdir(exist_ok=True)
cmd = [sys.executable, "/kaggle/working/gpu_worker.py", str(source), "--output-dir", str(output_dir), "--language", "pt", "--whisper-model", "tiny", "--semitones", str(SEMITONES)]
subprocess.run(cmd, check=True)
(output_dir / "source_reference.txt").write_text(f"Fonte: {source_label}\nTom: {SEMITONES:+d} semitons\n", encoding="utf-8")
archive = shutil.make_archive("/kaggle/working/karaoke_kaggle_output", "zip", output_dir)
print("Pacote pronto:", archive)
print("Conteúdo:", [p.name for p in output_dir.iterdir()])